# 03 探索性分析

本 Notebook 对问题一所需的门店、商品、趋势和星期效应做描述性统计。

## 代码说明：读取阶段 1 输出

这段代码读取 `daily_store_product_sales.csv`。由于负数量已确认为损耗/冲销类调整，问题一使用 `positive_sales` 作为销量目标。

In [ ]:
import pandas as pd
from src.config import PROJECT_ROOT

daily = pd.read_csv(PROJECT_ROOT / 'data/processed/daily_store_product_sales.csv', parse_dates=['date'])
daily['target_sales'] = daily['positive_sales'].astype(float)
daily.head()


输出怎么看：若能看到 `target_sales`，说明预测目标已按人工确认口径生成。

## 代码说明：门店销量统计

这段代码统计各门店累计销量和销售占比，用于判断门店差异。

In [ ]:
store_stats = daily.groupby(['store_id','store_name']).agg(total_sales=('target_sales','sum'), avg_daily_sales=('target_sales','mean')).reset_index().sort_values('total_sales', ascending=False)
store_stats['sales_share'] = store_stats['total_sales'] / store_stats['total_sales'].sum()
store_stats


输出怎么看：销量越高的门店对总需求贡献越大。若门店差异明显，预测时应按门店分别建模。

## 代码说明：商品销量统计

这段代码按商品编号统计销量。`11001020` 的两个名称按同一编号合并。

In [ ]:
product_stats = daily.groupby(['product_id','category']).agg(product_name=('product_name', lambda s: ' / '.join(sorted(set(map(str, s))))), total_sales=('target_sales','sum')).reset_index().sort_values('total_sales', ascending=False)
product_stats


输出怎么看：商品销量差异越大，越需要分别预测不同商品。

## 代码说明：日趋势和星期效应

这段代码分别聚合全体日销量和星期销量，用于观察趋势和周内规律。

In [ ]:
daily_total = daily.groupby('date', as_index=False)['target_sales'].sum()
daily_total['rolling_7d'] = daily_total['target_sales'].rolling(7, min_periods=1).mean()
weekday_stats = daily.groupby('weekday', as_index=False)['target_sales'].sum()
display(daily_total.head())
display(weekday_stats)


输出怎么看：7 日滚动均值用于看趋势；星期统计用于判断同星期均值模型是否合理。